# Hypothesis Testing

이 노트북은 [selected_smartfarm.csv](/c:/Users/ui203/OneDrive/문서/green_project/smartfarm_code/output/selected_smartfarm.csv)만 사용한다.

분석 원칙:
- `modeling_stronger_jun.ipynb`의 전처리 방향을 따른다.
- 1분 원본을 직접 해석하지 않고 10분 `tumbling mean` 집계 후 파생변수를 다시 계산한다.
- 현재 파일에는 `pump_on`, `lights_on`, `anomaly_type`, `cleaning_event_flag`가 없으므로 활성 구간은 `flow_rate_l_min > 1.0` 프록시로 정의한다.
- 먼저 `연속형 시계열끼리의 Granger / CCF`만 검정한다.

검정 대상:
- H1 원안: `zone1_resistance -> zone1_moisture_response_pct`
- H1 대체축: `filter_delta_p_kpa -> flow_drop_rate`
- CCF 축: `flow_rate_l_min <-> zone1_substrate_moisture_pct`

주의:
- 이 노트북은 현재 스키마에서 방어 가능한 검정만 수행한다.
- H3의 `세척 전후 T-test`는 라벨이 없으므로 여기서는 다루지 않는다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("")  # 파일 경로 입력

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df["timestamp"], df["drain_ec_ds_m"], color="#2196F3", linewidth=1.2, label="drain_ec_ds_m")
ax.axhline(df["drain_ec_ds_m"].min(), color="green", linestyle="--", linewidth=0.8, label=f"min: {df['drain_ec_ds_m'].min():.3f}")
ax.axhline(df["drain_ec_ds_m"].max(), color="red",   linestyle="--", linewidth=0.8, label=f"max: {df['drain_ec_ds_m'].max():.3f}")

ax.set_title("drain_ec_ds_m — 최소/최대 추이", fontsize=13)
ax.set_xlabel("timestamp")
ax.set_ylabel("EC (ds/m)")
ax.legend()
plt.tight_layout()
plt.savefig("drain_ec_plot.png", dpi=150)
plt.show()

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f as f_dist

ALPHA = 0.05
ACTIVE_FLOW_THRESHOLD = 1.0
GRANGER_MAX_LAG = 6
CCF_MAX_LAG = 12
EPS = 1e-6

CSV_PATH = Path('smartfarm_code/output/selected_smartfarm.csv')
OUT_DIR = Path('Analysis') / 'hypothesis_testing_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')


def load_raw_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, parse_dates=['timestamp'])
    return df.sort_values('timestamp').set_index('timestamp')


def aggregate_10m_tumbling(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return df[numeric_cols].resample('10min').mean().dropna()


def create_modeling_features_after_agg(df: pd.DataFrame) -> pd.DataFrame:
    feat = df.copy()
    dt_s = feat.index.to_series().diff().dt.total_seconds().fillna(600)

    feat['differential_pressure_kpa'] = feat['discharge_pressure_kpa'] - feat['suction_pressure_kpa']
    feat['filter_delta_p_kpa'] = feat['filter_pressure_in_kpa'] - feat['filter_pressure_out_kpa']
    feat['hydraulic_power_kw'] = (feat['flow_rate_l_min'] * feat['differential_pressure_kpa']) / 60000
    feat['wire_to_water_efficiency'] = feat['hydraulic_power_kw'] / (feat['motor_power_kw'] + EPS)

    feat['flow_baseline_l_min'] = feat['flow_rate_l_min'].rolling(6, min_periods=1).mean().shift(1)
    feat['flow_drop_rate'] = (feat['flow_baseline_l_min'] - feat['flow_rate_l_min']) / (feat['flow_baseline_l_min'] + EPS)

    feat['temp_slope_c_per_s'] = feat['motor_temperature_c'].diff() / dt_s
    rpm_mean = feat['pump_rpm'].rolling(3, min_periods=1).mean()
    feat['rpm_stability_index'] = np.abs(feat['pump_rpm'] - rpm_mean) / (rpm_mean + EPS)

    feat['zone1_resistance'] = feat['zone1_pressure_kpa'] / (feat['zone1_flow_l_min'] + EPS)
    feat['zone1_moisture_response_pct'] = feat['zone1_substrate_moisture_pct'].diff()
    feat['pid_error_ec'] = feat['mix_ec_ds_m'] - feat['mix_target_ec_ds_m']
    feat['salt_accumulation_delta'] = feat['drain_ec_ds_m'] - feat['mix_ec_ds_m']

    return feat.replace([np.inf, -np.inf], np.nan)


def granger_f_test(data: pd.DataFrame, x_col: str, y_col: str, max_lag: int = 6, difference: bool = True) -> pd.DataFrame:
    z = data[[x_col, y_col]].dropna().copy()
    if difference:
        z = z.diff().dropna()

    rows = []
    for lag in range(1, max_lag + 1):
        w = z.copy()
        for i in range(1, lag + 1):
            w[f'{y_col}_lag_{i}'] = w[y_col].shift(i)
            w[f'{x_col}_lag_{i}'] = w[x_col].shift(i)
        w = w.dropna()
        if len(w) < 80:
            continue

        y = w[y_col].to_numpy(dtype=float)
        x_restricted = np.column_stack([np.ones(len(w))] + [w[f'{y_col}_lag_{i}'].to_numpy(dtype=float) for i in range(1, lag + 1)])
        x_unrestricted = np.column_stack([np.ones(len(w))] + [w[f'{y_col}_lag_{i}'].to_numpy(dtype=float) for i in range(1, lag + 1)] + [w[f'{x_col}_lag_{i}'].to_numpy(dtype=float) for i in range(1, lag + 1)])

        beta_r, *_ = np.linalg.lstsq(x_restricted, y, rcond=None)
        beta_u, *_ = np.linalg.lstsq(x_unrestricted, y, rcond=None)
        rss_r = np.sum((y - x_restricted @ beta_r) ** 2)
        rss_u = np.sum((y - x_unrestricted @ beta_u) ** 2)

        df1 = lag
        df2 = len(w) - x_unrestricted.shape[1]
        f_stat = ((rss_r - rss_u) / df1) / (rss_u / df2)
        p_value = f_dist.sf(f_stat, df1, df2)

        rows.append({
            'lag_10m': lag,
            'lag_minutes': lag * 10,
            'n_obs': len(w),
            'f_stat': float(f_stat),
            'p_value': float(p_value),
            'significant': bool(p_value < ALPHA),
        })
    return pd.DataFrame(rows)


def ccf_curve(data: pd.DataFrame, x_col: str, y_col: str, max_lag: int = 12, difference: bool = False):
    z = data[[x_col, y_col]].dropna().copy()
    if difference:
        z = z.diff().dropna()

    x = z[x_col].to_numpy(dtype=float)
    y = z[y_col].to_numpy(dtype=float)
    if len(z) < max_lag + 20 or x.std() < 1e-9 or y.std() < 1e-9:
        return pd.DataFrame(columns=['lag_10m', 'lag_minutes', 'corr']), {'lag_10m': None, 'lag_minutes': None, 'corr': None}

    x = (x - x.mean()) / (x.std() + 1e-9)
    y = (y - y.mean()) / (y.std() + 1e-9)

    rows = []
    for lag in range(0, max_lag + 1):
        if lag == 0:
            corr = np.corrcoef(x, y)[0, 1]
        else:
            corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]
        rows.append({'lag_10m': lag, 'lag_minutes': lag * 10, 'corr': float(corr)})

    curve = pd.DataFrame(rows)
    peak = curve.loc[curve['corr'].abs().idxmax()].to_dict()
    return curve, peak


In [ ]:
df_raw = load_raw_data(CSV_PATH)
df_10m = aggregate_10m_tumbling(df_raw)
df_feat = create_modeling_features_after_agg(df_10m)
df_active = df_feat.loc[df_feat['flow_rate_l_min'] > ACTIVE_FLOW_THRESHOLD].copy()

print(f'Raw shape: {df_raw.shape}')
print(f'10-minute aggregated shape: {df_10m.shape}')
print(f'Feature table shape: {df_feat.shape}')
print(f'Active-window shape (flow_rate_l_min > {ACTIVE_FLOW_THRESHOLD}): {df_active.shape}')
print(f'Time range: {df_feat.index.min()} ~ {df_feat.index.max()}')

profile = pd.DataFrame([
    {'scope': 'all', 'rows': len(df_feat), 'flow_mean': df_feat['flow_rate_l_min'].mean(), 'zone1_flow_mean': df_feat['zone1_flow_l_min'].mean()},
    {'scope': 'active', 'rows': len(df_active), 'flow_mean': df_active['flow_rate_l_min'].mean(), 'zone1_flow_mean': df_active['zone1_flow_l_min'].mean()},
])
display(profile)

display(df_active[['flow_rate_l_min', 'zone1_flow_l_min', 'zone1_resistance', 'filter_delta_p_kpa', 'flow_drop_rate']].describe().T)


## 1. Granger Causality Test

두 축을 함께 본다.
- H1 원안: `zone1_resistance -> zone1_moisture_response_pct`
- H1 대체축: `filter_delta_p_kpa -> flow_drop_rate`

전체 구간과 활성 구간을 둘 다 계산한다. 결론이 활성 구간에만 성립하면, 그 조건을 해석에 명시해야 한다.


In [ ]:
granger_cases = [
    ('H1_original', 'zone1_resistance', 'zone1_moisture_response_pct'),
    ('H1_alternative', 'filter_delta_p_kpa', 'flow_drop_rate'),
]

granger_summary = []
granger_detail = {}

for scope_name, scope_df in [('all', df_feat), ('active', df_active)]:
    for label, x_col, y_col in granger_cases:
        result = granger_f_test(scope_df, x_col=x_col, y_col=y_col, max_lag=GRANGER_MAX_LAG, difference=True)
        granger_detail[(scope_name, label)] = result
        best = result.sort_values('p_value').iloc[0]
        granger_summary.append({
            'scope': scope_name,
            'hypothesis': label,
            'x': x_col,
            'y': y_col,
            'best_lag_10m': int(best['lag_10m']),
            'best_lag_minutes': int(best['lag_minutes']),
            'p_value': float(best['p_value']),
            'significant': bool(best['significant']),
        })

granger_summary_df = pd.DataFrame(granger_summary)
display(granger_summary_df)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.flatten()
for ax, ((scope_name, label), result) in zip(axes, granger_detail.items()):
    sns.lineplot(data=result, x='lag_minutes', y='p_value', marker='o', ax=ax)
    ax.axhline(ALPHA, color='red', linestyle='--', linewidth=1)
    ax.set_title(f'{scope_name} | {label}')
    ax.set_xlabel('lag (minutes)')
    ax.set_ylabel('p-value')

plt.tight_layout()
plt.savefig(OUT_DIR / 'granger_summary_selected_smartfarm.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Cross-Correlation Function

현재 스키마에는 `pump_on`, `drainage_ratio_pct`, `drain event`가 없으므로, CCF는 배지 수분 반응 프록시로 먼저 검정한다.

- 레벨 CCF: 장기 공통 추세를 잡을 수 있다.
- 차분 CCF: 보다 짧은 반응 지연을 본다.

해석은 차분 결과를 우선한다.


In [1]:
ccf_targets = [
    ('flow_rate_l_min', 'zone1_substrate_moisture_pct'),
    ('zone1_resistance', 'zone1_substrate_moisture_pct'),
]

ccf_summary = []
ccf_detail = {}

for x_col, y_col in ccf_targets:
    for mode_name, use_diff in [('level', False), ('diff', True)]:
        curve, peak = ccf_curve(df_active, x_col=x_col, y_col=y_col, max_lag=CCF_MAX_LAG, difference=use_diff)
        ccf_detail[(x_col, y_col, mode_name)] = curve
        ccf_summary.append({
            'x': x_col,
            'y': y_col,
            'mode': mode_name,
            'peak_lag_10m': peak['lag_10m'],
            'peak_lag_minutes': peak['lag_minutes'],
            'peak_corr': peak['corr'],
        })

ccf_summary_df = pd.DataFrame(ccf_summary)
display(ccf_summary_df)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.flatten()
for ax, ((x_col, y_col, mode_name), curve) in zip(axes, ccf_detail.items()):
    sns.lineplot(data=curve, x='lag_minutes', y='corr', marker='o', ax=ax)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{mode_name} | {x_col} vs {y_col}')
    ax.set_xlabel('lag (minutes)')
    ax.set_ylabel('correlation')

plt.tight_layout()
plt.savefig(OUT_DIR / 'ccf_summary_selected_smartfarm.png', dpi=150, bbox_inches='tight')
plt.show()


NameError: name 'ccf_curve' is not defined

## 3. Interpretation

이 노트북에서 바로 사용할 수 있는 해석 규칙:
- `filter_delta_p_kpa -> flow_drop_rate`가 전체/활성 구간 모두 유의하면 메인 막힘 가설의 주된 통계 근거로 사용한다.
- `zone1_resistance -> zone1_moisture_response_pct`가 활성 구간에서만 유의하면, `관수 중일 때만 성립하는 국소 반응`으로 제한해서 기술한다.
- CCF는 차분 결과를 우선하고, 레벨 CCF는 보조 참고로만 쓴다.

현재 제한사항:
- `selected_smartfarm.csv`에는 `pump_on`, `lights_on`, `anomaly_type`, `cleaning_event_flag`가 없으므로 이벤트 기반 비교 검정은 불가하다.
- 따라서 H3의 세척 전후 T-test는 별도 라벨 파일 없이 수행하지 않는다.
